<a href="https://colab.research.google.com/github/Admindatosgobes/Laboratorio-de-Datos/blob/main/Siniestralidad_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizaciones de siniestralidad en carretera en 2024, Direccion General de Trafico

---



## 0. Introducción

Los datos de siniestralidad en carretera representan **una de las grandes preocupaciones** tanto institucionales como sociales, datos que siempre apelan tanto a la responsabilidad individual como colectiva, y que gracias a la Direccion General de Tráfico y su participación a través del portal de datos abiertos Datos.gob.es se pueden analizar y visualizar de forma intuitiva.

A partir de los datos disponibles para el año 2024 es posible **extraer métricas cuantitativas** para hacernos una idea de las situaciones y consecuencias de los accidentes de tráfico a lo largo del tiempo.

En la base de datos encontramos más de cien mil accidentes **agregados por días de la semana**, con información respecto al número total de víctimas (fallecidos, heridos graves o heridos leves en horizontes temporales tanto de 24h como de 30 días) y respecto al **tipo de accidente, tipo de vía, titularidad de la vía y municipio**.

En este ejericio realizaremos una serie de operaciones sencillas utilizando Python para poder extraer las métricas de nuestro interés en formato JSON, para su posterior visualización en Javascript, en concreto en D3.js.

El objetivo de este ejercicio es la **visualización de varias de las variables** que caracterizan la siniestralidad en carretera. Para ello el ejericio se estructura en tres pasos en esta etapa a desarrollar en Python:

1.   **Lectura** del fichero de datos de la DGT
   *    Creación de un **Dataframe**
   *    Creacion de una **fecha para cada entrada**


2.   **Cálculo** de las métricas para caracterizar los datos disponibles
*    Suma del **total de víctimas** en cada hora del día
*    Suma de **accidentes por categoría**
*    Asignación de valores a los **municipios**

3.   **Exportar** las métricas en formato JSON




## 1. Lectura del fichero de datos

### 1.1 Creación de un Dataframe

Para la lectura de los datos utilizaremos la libreria **Pandas** de Python, que nos permite estructurar los datos en filas y columnas de forma intuitiva y operar sobre ellos.  

Igualmente, y dado que tenemos la fecha en formato de año, mes y día de la semana así como la hora del accidente, necesitaremos una librería para modificar y crear formatos de fecha que podamos manejar fácilmente. Para ello disponemos de la librería **Datetime** en Python.

La libreria **IO** y **files** de Google Colab nos permite hacer una lectura de datos desde otras localizaciones fuera de este notebook

Por último para la parte geográfica de asignación de valores a los diferentes municipios necesitaremos una librería para poder procesar geometrías en forma de polígonos, y lo haremos a través de **Geopandas**.

In [ ]:
import pandas as pd
import numpy as np
import datetime
import geopandas as gpd
import io
from google.colab import files
import warnings
warnings.filterwarnings("ignore")

In [ ]:
uploaded = files.upload()

Saving TABLA_ACCIDENTES_24.csv to TABLA_ACCIDENTES_24 (1).csv


Procedemos entonces a la lectura del fichero de entrada en forma de pandas a partir de un fichero en formato .csv. Es importante utilizar el encoding del texto adecuado para una correcta interpretacion de determinados caracteres. En este caso, 'latin-1' es el adecuado dado que los datos que contiene están mayormente en castellano y asi la ortografía latina será bien interpretada por Python.
Una vez importados los datos y almacenados en un dataframe podemos ver el nombre de las variables y su tipo a través del comando `.info()` de Pandas, que nos ofrece:


*   **Número de entradas** en el dataframe
*   **Nombre de las columnas** asociada con cada variable
*   **Tipo de cada variable**, incluyendo string, float o int.



In [ ]:
content = pd.read_csv(io.BytesIO(uploaded['TABLA_ACCIDENTES_24 (1).csv']), encoding='latin-1')
content.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101996 entries, 0 to 101995
Data columns (total 73 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID_ACCIDENTE              101996 non-null  int64  
 1   ANYO                      101996 non-null  int64  
 2   MES                       101996 non-null  int64  
 3   DIA_SEMANA                101996 non-null  int64  
 4   HORA                      101996 non-null  int64  
 5   COD_PROVINCIA             101996 non-null  int64  
 6   COD_MUNICIPIO             101996 non-null  int64  
 7   ISLA                      4444 non-null    float64
 8   ZONA                      101996 non-null  int64  
 9   ZONA_AGRUPADA             101996 non-null  int64  
 10  CARRETERA                 101996 non-null  object 
 11  KM                        38814 non-null   float64
 12  SENTIDO_1F                101996 non-null  int64  
 13  TITULARIDAD_VIA           101996 non-null  i

Una vez ejecutada la función `.info()` podemos empezar a operar sobre la estructura y las variables contenidas en el dataframe.

### 1.2 Creación de una fecha para cada entrada

La información temporal de los datos viene en **cuatro columnas separadas** para el año, el mes, el día de la semana (de lunes a domingo), y por último la hora del accidente.
Dado que los datos está agregados en origen por día de la semana y no por día natural, intentaremos crear una serie temporal desde una perspectiva semanal.
Para ello son necesarias una serie de transformaciones, en las cuales **crearemos primero una columna con la fecha y hora** de cada accidente de forma más compacta que cuatro columnas por separado.
En primer lugar, reasignamos el nombre de cada columna a los estándares de la librería Datetime de Python.

In [ ]:
content = content.rename(columns={'ANYO': 'year', 'MES': 'month', 'DIA_SEMANA': 'day','HORA':'hour'})

Como realizaremos un analisis semanal de la ocurrencia de accidentes, asignamos valores constantes tanto al año como al mes, ya que queremos ver la variabilidad semanal y no estacional. De esta forma podremos superponer diferentes días de la semana sobre la misma escala temporal de mes y año.
Para ello usaremos la función lambda que opera sobre todos los elementos de una columna de un dataframe, asignándoles el valor que queramos.

In [ ]:
content['year']= content['year'].apply(lambda x: 2024)
content['month']= content['month'].apply(lambda x: 1)

Seguidamente creamos una columna con la fecha en formato `datetime` e igualmente creamos otra columna con la hora en formato `datetime` a través de la función `.to_datetime()`

In [ ]:
content['date'] = pd.to_datetime(content[['year', 'month', 'day']])
content['hour'] = pd.to_datetime(content['hour'], format='%H')

Para poder crear la columna definitiva con la fecha y la hora, convertimos ambas columnas en `string` a través de la función de datetime (dt) para tal efecto, `.strftime()`

In [ ]:
content['hour'] = content['hour'].dt.strftime('%H:%M:%S')
content['date'] = content['date'].dt.strftime('%m-%d-%y')

Y concatenamos el contenido con el símbolo de la suma y lo reconvertimos a formato `datetime` tal y como vimos anteriormente.

In [ ]:
content['date'] = pd.to_datetime(content['date'] + ' ' + content['hour'])
content['date']

,date
0,2024-01-03 22:00:00
1,2024-01-06 23:00:00
2,2024-01-07 01:00:00
3,2024-01-07 08:00:00
4,2024-01-04 17:00:00
...,...
101991,2024-01-01 14:00:00
101992,2024-01-06 10:00:00
101993,2024-01-05 08:00:00
101994,2024-01-02 18:00:00


Aquí observamos cómo el dataframe tiene ahora una columna en formato de fecha estándar tanto para Pandas como para Python, sobre el cual ya podemos operar.

## 2. Cálculo de las Métricas

### 2.1 Suma del total de víctimas en cada hora del día

El número total de víctimas es el más grande dentro de toda la estadística de víctimas (fallecidos, heridos graves y heridos leves), y por cantidad es el que nos va a revelar de forma más precisa los patrones de los accidentes de tráfico en el país.  
Podemos por lo tanto calcular el número total de víctimas que se han producido en cada una de las horas del día. Como ya hemos fijado el año y el mes, sólo queda calcular la suma sobre cada una de las horas. Para ello utilizamos la funcion .duplicate() de pandas, que nos permite identificar aquellos valores de la fecha duplicadas sobre las cuales operará la suma.


In [ ]:
content_dups = content[content.duplicated(['date'], keep=False)].copy()

El cálculo se realiza ahora sobre el número total de víctimas computadas a las 24h, y adjuntamos al nuevo dataframe las columnas correspondientes víctimas totales a 24h.

In [ ]:
content_sum = content_dups.groupby('date')['TOTAL_VICTIMAS_24H'].apply(lambda x : x.sum())
content_sum = content_sum.reset_index()
content_sum

,date,TOTAL_VICTIMAS_24H
0,2024-01-01 00:00:00,276
1,2024-01-01 01:00:00,224
2,2024-01-01 02:00:00,152
3,2024-01-01 03:00:00,93
4,2024-01-01 04:00:00,114
...,...,...
163,2024-01-07 19:00:00,1142
164,2024-01-07 20:00:00,993
165,2024-01-07 21:00:00,701
166,2024-01-07 22:00:00,695


De esta forma tenemos la suma del total de víctimas de todas aquellas entradas de fecha que coinciden, realizando así la suma por hora y día de la semana.

Para agregar nuestros resultados por hora del día, creamos una nueva columna con la información del día de la semana asociado a cada accidente. Esto nos permitirá crear ficheros de salida correspondientes a cada día de la semana. Primero creamos la columna con la información del día en formato `string`:

In [ ]:
content_sum['dayz']=content_sum['date'].dt.strftime('%d')

Y luego vemos qué valores únicos tenemos.

In [ ]:
dayz = content_sum['dayz'].unique().tolist()
dayz

['01', '02', '03', '04', '05', '06', '07']

Una vez tenemos los valores únicos, podemos hacer un bucle sobre todos los días de la semana para asignarles el mismo día y poder superponer más adelante los valores sobre la misma gráfica. Y hacemos tantos ficheros de salida como días de la semana para tener cada día por separado de forma clara y concisa.
En este punto es importante indicar el formato de salida de la fecha como `iso` para su posterior uso en Javascript.

In [ ]:
for i in dayz:
    content_sum_dayz = content_sum.loc[content_sum['dayz']==i]
    content_sum_dayz['date'] = content_sum_dayz['date'].apply(lambda dt: dt.replace(day=1))
    # content_sum_dayz.reset_index().to_json(date_format='iso',orient='records',path_or_buf=r'C:\Users\JorgeMartinezRey\Documents\Red\DATA\Siniestros_Serie_Temporal_'+str(i)+'.json')

Por último, y para encuadrar los resultados de la siniestralidad por hora para cada día de la semana dentro de toda la serie semanal, calculamos el máximo y el mínimo del total de víctimas para cada hora.
Para ello primero asignamos el mismo día de la semana a todos los accidentes

In [ ]:
content_sum['date']=content_sum['date'].apply(lambda dt: dt.replace(day=1))

Después creamos un array con todas las horas únicas en las cuales hay registrados accidentes, es decir, todas las horas del día

In [ ]:
unc = content_sum['date'].unique()
unc

<DatetimeArray>
['2024-01-01 00:00:00', '2024-01-01 01:00:00', '2024-01-01 02:00:00',
 '2024-01-01 03:00:00', '2024-01-01 04:00:00', '2024-01-01 05:00:00',
 '2024-01-01 06:00:00', '2024-01-01 07:00:00', '2024-01-01 08:00:00',
 '2024-01-01 09:00:00', '2024-01-01 10:00:00', '2024-01-01 11:00:00',
 '2024-01-01 12:00:00', '2024-01-01 13:00:00', '2024-01-01 14:00:00',
 '2024-01-01 15:00:00', '2024-01-01 16:00:00', '2024-01-01 17:00:00',
 '2024-01-01 18:00:00', '2024-01-01 19:00:00', '2024-01-01 20:00:00',
 '2024-01-01 21:00:00', '2024-01-01 22:00:00', '2024-01-01 23:00:00']
Length: 24, dtype: datetime64[ns]

A continuación creamos dos arrays vacíos donde vamos a almacenar el mínimo y el máximo del total de víctimas para cada hora

In [ ]:
themin=[]
themax=[]

Y por último hacemos un bucle sobre todas las horas del día e incorporamos a los arrays que hemos creado el cálculo del máximo y el mínimo en esa hora.

In [ ]:
for t in unc:
    content_slice = content_sum.loc[content_sum['date']==t]
    themin.append(np.min(content_slice['TOTAL_VICTIMAS_24H']))
    themax.append(np.max(content_slice['TOTAL_VICTIMAS_24H']))

Una vez calculados los máximos y los mínimos, creamos un dataframe para la posterior creación de un fichero JSON con los resultados

In [ ]:
uncertainties = pd.DataFrame({'date':unc,'min':themin,'max':themax})
uncertainties

,date,min,max
0,2024-01-01 00:00:00,221,577
1,2024-01-01 01:00:00,143,454
2,2024-01-01 02:00:00,81,338
3,2024-01-01 03:00:00,61,312
4,2024-01-01 04:00:00,73,298
5,2024-01-01 05:00:00,158,388
6,2024-01-01 06:00:00,303,509
7,2024-01-01 07:00:00,387,932
8,2024-01-01 08:00:00,398,1247
9,2024-01-01 09:00:00,526,1196


### 2.2 Suma del número de accidentes por categoría

En esta sección procedemos a calcular la cantidad de incidencias que hay para cada tipo de accidente y para las diferentes titularidades de la vía. El mecanismo es el mismo para las dos, utilizando la función `.value_counts()` de Pandas.

Para el caso del tipo de accidente tenemos:

In [ ]:
stats_tipo = content['TIPO_ACCIDENTE'].value_counts().reset_index()
stats_tipo

,TIPO_ACCIDENTE,count
0,2.0,22176
1,4.0,17279
2,7.0,12491
3,3.0,8851
4,10.0,8454
5,19.0,4805
6,20.0,4618
7,9.0,3664
8,1.0,3329
9,16.0,3287


Para el caso de la titularidad de la vía:

In [ ]:
stats_titularidad = content['TITULARIDAD_VIA'].value_counts().reset_index()
stats_titularidad

,TITULARIDAD_VIA,count
0,4,50288
1,5,22446
2,1,11739
3,2,10525
4,3,6970
5,999,28


Para el caso del tipo de vía:

In [ ]:
stats_via = content['TIPO_VIA'].value_counts().reset_index()
stats_via

,TIPO_VIA,count
0,9,48432
1,6,20281
2,14,17519
3,3,8136
4,2,3316
5,5,1769
6,10,1055
7,1,425
8,7,384
9,8,322


Una vez tenemos estas variables las exportamos de la misma forma que lo hemos hecho anteriormente con la incidencia semanal, utilizando la función `.to_json()`

In [ ]:
#stats_tipo.reset_index().to_json(orient='records',path_or_buf=r'C:\Users\JorgeMartinezRey\Documents\Red\DATA\Siniestros_Tipo_de_Via.json')
#stats_titularidad.reset_index().to_json(orient='records',path_or_buf=r'C:\Users\JorgeMartinezRey\Documents\Red\DATA\Siniestros_Titularidad_de_Via.json')
#stats_via.reset_index().to_json(orient='records',path_or_buf=r'C:\Users\JorgeMartinezRey\Documents\Red\DATA\Siniestros_Urbana.json')

### 2.3 Asignación de valores a los municipios

Para la cartografía de la densidad de accidentes por municipios recurrimos tanto a los datos del municipio donde se ha producido el accidente como a la topografía de los diferentes municipios, a los que les asignaremos el valor de las estadísticas de las que disponemos.

Particularizamos este mapa para un caso concreto como ejemplo, tomando la provincia de Valencia. Con los datos públicos de los que disponemos, importamos la geometría de todos los municipios de la provincia.

In [ ]:
uploaded = files.upload()

Saving ca_municipios_20251210.geojson to ca_municipios_20251210.geojson


Una vez disponemos del archivo, lo abrimos y leemos como pandas utilizando la librería de geopandas, con su característica columna de `geometry`, donde se especifica la geometría en forma de puntos, líneas o polígonos.

En geopandas también es fácil de cambiar el sistema de coordenadas de referencia. Dado que la amplia mayoría de herramientas de visualización en formato web utiliza el estándar de EPSG:4326, asignamos a los datos entrantes tal formato con la función `.to_crs(4326)`.

In [ ]:
import geopandas as gpd
gdf = gpd.read_file(io.BytesIO(uploaded['ca_municipios_20251210.geojson']))
gdf = gdf.to_crs(4326)
gdf

,MUNIINE,MUNI_COD,MUNI_KEY,CODPROV,CODAUTO,CODCOMAR,NOMBRE,geometry
0,03001,10803001,141,03,12,10,"ATZUBIA, L'","POLYGON ((-0.19632 38.85944, -0.19594 38.86003..."
1,03002,10803002,142,03,12,05,AGOST,"POLYGON ((-0.70576 38.45214, -0.68783 38.4595,..."
2,03003,10803003,143,03,12,07,AGRES,"POLYGON ((-0.54826 38.80565, -0.54745 38.8056,..."
3,03004,10803004,144,03,12,09,AIGUES,"POLYGON ((-0.39296 38.49175, -0.39256 38.49295..."
4,03005,10803005,145,03,12,01,ALBATERA,"POLYGON ((-0.944 38.25543, -0.94317 38.25509, ..."
...,...,...,...,...,...,...,...,...
537,46262,10846262,7239,46,12,03,YESA (LA),"POLYGON ((-0.98153 39.83975, -0.98144 39.84653..."
538,46263,10846263,7240,46,12,32,ZARRA,"POLYGON ((-1.21398 39.12319, -1.21127 39.12324..."
539,46902,10846902,7241,46,12,00,GATOVA,"POLYGON ((-0.58132 39.75672, -0.57715 39.76802..."
540,46903,10846903,104971830875810211,46,12,00,SAN ANTONIO DE BENAGEBER,"POLYGON ((-0.52159 39.57613, -0.52131 39.57631..."


A continuación agregamos tal y como hicimos con las cantidades cualitativas, las métricas en torno a los valores únicos de municipio:

In [ ]:
stats_codigo = content['COD_MUNICIPIO'].value_counts().reset_index()
stats_codigo = stats_codigo.sort_values('COD_MUNICIPIO',ascending=True)
stats_codigo = stats_codigo.reset_index()
stats_codigo

,index,COD_MUNICIPIO,count
0,0,0,12232
1,650,1002,17
2,893,1036,10
3,839,1051,12
4,18,1059,547
...,...,...,...
1310,513,50272,23
1311,8,50297,1095
1312,531,50298,22
1313,50,51001,300


Tenemos un total de 1315 municipios diferentes en los que ha habido accidentes en 2024. Para su posterior filtrado para la provincia de Valencia, procedemos a cambiar el formato, siempre con número de dígitos igual a cinco, de la siguiente forma, primero cambiamos a formato string y luego completamos con ceros.

In [ ]:
stats_codigo['COD_MUNICIPIO'] =  stats_codigo['COD_MUNICIPIO'].astype(str)
stats_codigo['COD_MUNICIPIO'] = stats_codigo['COD_MUNICIPIO'].str.zfill(5)

De esta forma nos es fácil leer los dos primeros dígitos de esos valores, y seleccionar aquellos que empiezan por 46, que es el código de la provincia de Valencia. Esta técnica se aplica tanto a los datos de accidentes como a las geometrías de municipios que acabamos de leer.

In [ ]:
stats_codigo['front'] = stats_codigo['COD_MUNICIPIO'].apply(lambda x: x[:2])
valencia = stats_codigo[stats_codigo['front'] == '46']
gdf['front'] = gdf['MUNIINE'].apply(lambda x: x[:2])
gdf_valencia = gdf[gdf['front'] == '46']

Por último, construimos un dataframe con aquellos municipios que están tanto en los datos de accidentes como de aquellos para los cuales tenemos geometrías definidas. Y a ese nuevo dataframe les añadimos el resto de municipios de la provincia de Valencia, ya con valor 0 al no estar reflejados en la estadística de accidentes. Así conseguimos tener al menos un valor asociado a cada municipio.

In [ ]:
list_muni = valencia['COD_MUNICIPIO'].tolist()

gdf_valencia_short = gdf_valencia.loc[gdf_valencia['MUNIINE'].isin(list_muni)]
gdf_valencia_short['count']=valencia['count'].values

gdf_valencia_long = gdf_valencia.loc[~gdf_valencia['MUNIINE'].isin(list_muni)]
gdf_valencia_long['count']=0

gdf_valencia_all = pd.concat([gdf_valencia_short, gdf_valencia_long], axis=0, ignore_index=True)
gdf_valencia_all

,MUNIINE,MUNI_COD,MUNI_KEY,CODPROV,CODAUTO,CODCOMAR,NOMBRE,geometry,front,count
0,46005,10846005,6982,46,12,00,ALAQUAS,"POLYGON ((-0.50769 39.44455, -0.50533 39.44524...",46,40
1,46006,10846006,6983,46,12,26,ALBAIDA,"POLYGON ((-0.57798 38.8696, -0.57834 38.87057,...",46,7
2,46007,10846007,6984,46,12,18,ALBAL,"POLYGON ((-0.45013 39.40068, -0.4492 39.40085,...",46,51
3,46011,10846011,6988,46,12,30,ALBERIC,"POLYGON ((-0.58502 39.11954, -0.5839 39.12123,...",46,10
4,46013,10846013,6990,46,12,13,ALBORAIA/ALBORAYA,"POLYGON ((-0.35972 39.50342, -0.35972 39.50341...",46,92
...,...,...,...,...,...,...,...,...,...,...
261,46261,10846261,7238,46,12,02,YATOVA,"POLYGON ((-0.98278 39.3367, -0.98175 39.34038,...",46,0
262,46262,10846262,7239,46,12,03,YESA (LA),"POLYGON ((-0.98153 39.83975, -0.98144 39.84653...",46,0
263,46263,10846263,7240,46,12,32,ZARRA,"POLYGON ((-1.21398 39.12319, -1.21127 39.12324...",46,0
264,46902,10846902,7241,46,12,00,GATOVA,"POLYGON ((-0.58132 39.75672, -0.57715 39.76802...",46,0


Por último, exportamos el resultado en formato GeoJSON para su posterior visualización en Javascript

In [ ]:
#gdf_valencia_all.to_file(r'\Users\JorgeMartinezRey\Documents\Red\DATA\DGT_valencia.geojson',driver='GeoJSON')